# 06 — Bias Calibration / ปรับ bias หลังเทรน

**EN.** Fits a per-(station, horizon) bias calibration on the validation-fold predictions cached by notebook `03_train_baseline.ipynb` (and optionally `04_train_quantile.ipynb`). The calibration helper lives in `app/core/calibration.py` and auto-picks the best level from `{L1, L2, L3, L4}` via val MAE. Output is written next to each v3 model as `app/models/forecast_v3/{station}/h{H}/calibration.json` — `bundle.json` is **not** modified.

**TH.** สมุดงานนี้ฟิตค่าชดเชย bias ต่อ (station, horizon) จากผลพยากรณ์บน validation fold ที่ notebook `03_train_baseline.ipynb` (และเสริมด้วย `04_train_quantile.ipynb`) บันทึกไว้. ตัว helper อยู่ที่ `app/core/calibration.py` และจะเลือกระดับ `{L1, L2, L3, L4}` ที่ให้ val MAE ต่ำสุดให้อัตโนมัติ. ไฟล์ผลลัพธ์จะถูกเขียนคู่กับ v3 model ที่ `app/models/forecast_v3/{station}/h{H}/calibration.json` โดย**ไม่แตะ** `bundle.json`.

## Path contract / สัญญาเส้นทางของไฟล์

**Notebook 06 expects** notebook 03 (and optionally 04) to have written one parquet per (station, horizon) at:

```
notebooks/colab/_artifacts/val_predictions/{station_id}__h{H}.parquet
```

with at minimum the columns:

| column         | dtype           | meaning                                                     |
|----------------|-----------------|-------------------------------------------------------------|
| `station_id`   | str             | one of the five TMD stations                                |
| `horizon_h`    | int             | forecast horizon in hours (must match the filename)         |
| `ts_utc`       | datetime64[UTC] | timestamp of the **target** (i.e. ts_obs + horizon_h)       |
| `y_true`       | float           | observed heat-index at `ts_utc`                              |
| `y_pred`       | float           | model val-fold prediction (mean / q50 / etc.)                |
| `hour_local`   | int (0–23)      | local hour-of-day for the target timestamp (Asia/Bangkok)    |

If that path does not exist, this notebook raises a clear error pointing back to notebook 03. **Follow-up note:** `app/ml/forecast/predict.py` will need a small change to load `calibration.json` and pipe predictions through `Calibration.apply(...)`; that change is **out of scope** for this notebook.

**TH.** notebook 06 คาดว่า notebook 03 (และ 04 ถ้ามี) ได้บันทึก parquet ของ val predictions ไว้ที่ `notebooks/colab/_artifacts/val_predictions/{station_id}__h{H}.parquet` ตามคอลัมน์ข้างต้น. หากไม่มีไฟล์ดังกล่าว notebook นี้จะ raise error อย่างชัดเจน. ส่วน `predict.py` จะต้องแก้เล็กน้อยเพื่ออ่าน `calibration.json` และเรียก `Calibration.apply(...)` ตอน inference — **ไม่อยู่ในขอบเขต**ของ notebook นี้.

In [ ]:
# --- bootstrap (re-run if you opened this notebook before 00_setup) -------
import os, sys
REPO_DIR = "/content/Heat-wave-backend"
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
elif not os.path.isdir(REPO_DIR):
    # local dev fallback — assume the notebook lives in <repo>/notebooks/colab/
    here = os.getcwd()
    if os.path.basename(here) == "colab":
        REPO_DIR = os.path.abspath(os.path.join(here, "..", ".."))
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print("cwd      =", os.getcwd())

from app.core.calibration import (
    Calibration,
    fit_calibration,
    select_calibration,
)
from app.data.stations import STATIONS
print("stations:", list(STATIONS.keys()))

## 1 — Load val predictions from notebook 03 / โหลดผลทำนาย val

**EN.** We scan `notebooks/colab/_artifacts/val_predictions/` for files named `{station}__h{H}.parquet`. Missing files raise a `FileNotFoundError` with the exact list of expected paths so you know which station/horizon still needs to be re-run in notebook 03.

**TH.** เราจะอ่านโฟลเดอร์ `notebooks/colab/_artifacts/val_predictions/` ตามชื่อ `{station}__h{H}.parquet`. ถ้าหาไม่เจอ จะ raise `FileNotFoundError` พร้อมแสดงรายการไฟล์ที่ต้องมี เพื่อให้ย้อนกลับไปรัน notebook 03 ได้ตรงจุด.

In [ ]:
import pandas as pd
from pathlib import Path

VAL_PRED_DIR = Path("notebooks/colab/_artifacts/val_predictions")
EXPECTED_HORIZONS = [6, 12, 24, 48, 72]
EXPECTED_STATIONS = list(STATIONS.keys())

REQUIRED_COLS = {"station_id", "horizon_h", "ts_utc", "y_true", "y_pred", "hour_local"}

if not VAL_PRED_DIR.exists():
    raise FileNotFoundError(
        f"Validation-prediction cache not found at {VAL_PRED_DIR.resolve()}.\n"
        "Notebook 03_train_baseline.ipynb must write one parquet per (station, horizon) "
        "following the path contract documented in cell 0 of this notebook."
    )

expected_paths = [
    VAL_PRED_DIR / f"{sid}__h{h}.parquet"
    for sid in EXPECTED_STATIONS
    for h in EXPECTED_HORIZONS
]
found = [p for p in expected_paths if p.exists()]
missing = [p for p in expected_paths if not p.exists()]

print(f"Found {len(found)} / {len(expected_paths)} expected val-prediction files.")
if missing:
    print(f"Missing {len(missing)} files — calibration will skip those (station, horizon) pairs:")
    for p in missing:
        print("  -", p)

if not found:
    raise FileNotFoundError(
        f"No val-prediction parquet files found under {VAL_PRED_DIR.resolve()}.\n"
        "Re-run notebook 03_train_baseline.ipynb so it writes "
        "`{station}__h{H}.parquet` files into that directory."
    )

frames = []
for p in found:
    df_p = pd.read_parquet(p)
    missing_cols = REQUIRED_COLS - set(df_p.columns)
    if missing_cols:
        raise ValueError(
            f"{p} is missing required columns: {missing_cols}. "
            "Update notebook 03 to emit the contract documented at the top of this notebook."
        )
    frames.append(df_p[list(REQUIRED_COLS)])

val_df = pd.concat(frames, ignore_index=True)
val_df["horizon_h"] = val_df["horizon_h"].astype(int)
val_df["hour_local"] = val_df["hour_local"].astype(int)
val_df["station_id"] = val_df["station_id"].astype(str)
print(f"\nLoaded {len(val_df):,} rows across {val_df['station_id'].nunique()} stations "
      f"and {val_df['horizon_h'].nunique()} horizons.")
val_df.head()

### 1b — Time-based split for fit vs select / แบ่ง val ออกเป็น fit และ select

**EN.** `select_calibration` needs separate fit vs evaluation arrays. We split each (station, horizon) by time: the earlier 70% trains the calibration, the latest 30% selects the level. This avoids fitting and selecting on identical rows.

**TH.** `select_calibration` ต้องใช้ array สำหรับ fit และ select แยกกัน เราตัดตามเวลาในแต่ละ (station, horizon): 70% แรกใช้ fit, 30% ท้ายใช้เลือก level.

In [ ]:
import numpy as np

FIT_FRACTION = 0.70

val_df = val_df.sort_values(["station_id", "horizon_h", "ts_utc"]).reset_index(drop=True)
split_flags = np.zeros(len(val_df), dtype=bool)  # True → in fit set
for (sid, h), group in val_df.groupby(["station_id", "horizon_h"], sort=False):
    n = len(group)
    cut = max(1, int(n * FIT_FRACTION))
    split_flags[group.index[:cut]] = True

fit_df = val_df.loc[split_flags].reset_index(drop=True)
sel_df = val_df.loc[~split_flags].reset_index(drop=True)
print(f"fit rows: {len(fit_df):,}    select rows: {len(sel_df):,}")

## 2 — Fit per-(station, horizon) calibration / ฟิต calibration ทุกคู่

**EN.** For each (station, horizon) we call `select_calibration(...)`, which fits L1–L4 and returns the level with the lowest MAE on the held-out 30% slice. We then re-apply the chosen calibration to the **full** val rows for that pair to compute the final raw-vs-calibrated metrics.

**TH.** สำหรับแต่ละ (station, horizon) เรียก `select_calibration(...)` ซึ่งจะฟิต L1–L4 แล้วเลือกระดับที่ MAE ต่ำสุดบน 30% ส่วนปลาย จากนั้น apply กลับเข้ากับ val ทั้งหมดของคู่นั้นเพื่อคำนวณตัวเลขเปรียบเทียบ raw vs calibrated.

In [ ]:
def _mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def _bias(y_true, y_pred):
    return float(np.mean(np.asarray(y_pred) - np.asarray(y_true)))

calibrations: dict[tuple[str, int], Calibration] = {}
rows = []
calibrated_full = np.full(len(val_df), np.nan)

for (sid, h), group_full in val_df.groupby(["station_id", "horizon_h"], sort=False):
    g_fit = fit_df[(fit_df.station_id == sid) & (fit_df.horizon_h == h)]
    g_sel = sel_df[(sel_df.station_id == sid) & (sel_df.horizon_h == h)]
    if len(g_fit) < 30 or len(g_sel) < 10:
        print(f"  [skip] {sid} h={h}: too few rows (fit={len(g_fit)}, sel={len(g_sel)})")
        continue

    calib = select_calibration(
        y_true_train=g_fit.y_true.values,
        y_pred_train=g_fit.y_pred.values,
        station_ids_train=g_fit.station_id.values,
        horizons_train=g_fit.horizon_h.values,
        hours_train=g_fit.hour_local.values,
        y_true_val=g_sel.y_true.values,
        y_pred_val=g_sel.y_pred.values,
        station_ids_val=g_sel.station_id.values,
        horizons_val=g_sel.horizon_h.values,
        hours_val=g_sel.hour_local.values,
    )
    calibrations[(sid, h)] = calib

    y_pred_cal = calib.apply(
        group_full.y_pred.values,
        group_full.station_id.values,
        group_full.horizon_h.values,
        group_full.hour_local.values,
    )
    calibrated_full[group_full.index.values] = y_pred_cal

    raw_mae = _mae(group_full.y_true, group_full.y_pred)
    cal_mae = _mae(group_full.y_true, y_pred_cal)
    raw_b = _bias(group_full.y_true, group_full.y_pred)
    cal_b = _bias(group_full.y_true, y_pred_cal)
    rows.append({
        "station_id": sid,
        "horizon_h": h,
        "level": calib.level,
        "raw_mae": raw_mae,
        "cal_mae": cal_mae,
        "mae_delta": cal_mae - raw_mae,
        "raw_bias": raw_b,
        "cal_bias": cal_b,
        "n_val": len(group_full),
    })
    print(f"  {sid} h={h:>2d}  level={calib.level}  raw_mae={raw_mae:.3f}  cal_mae={cal_mae:.3f}  Δ={cal_mae-raw_mae:+.3f}")

summary = pd.DataFrame(rows).sort_values(["station_id", "horizon_h"]).reset_index(drop=True)
val_df["y_pred_cal"] = calibrated_full
print(f"\nFitted calibrations for {len(calibrations)} (station, horizon) pairs.")

## 3 — Save `calibration.json` per (station, horizon) / บันทึกไฟล์

**EN.** We write `app/models/forecast_v3/{station}/h{H}/calibration.json` using `Calibration.to_json()`. We do **not** touch `bundle.json`. If the v3 slot directory does not yet exist (i.e. the model has not been trained for that pair), the calibration is skipped with a warning.

**TH.** เขียนไฟล์ `calibration.json` ด้วย `Calibration.to_json()` ลงในโฟลเดอร์ของ v3 model ของแต่ละคู่. **ไม่แตะ** `bundle.json`. ถ้ายังไม่ได้เทรน v3 ของคู่นั้น (ไม่มีโฟลเดอร์) จะข้ามและเตือนเฉย ๆ.

> **Follow-up reminder / สิ่งที่ต้องทำต่อ:** `app/ml/forecast/predict.py` will need a small patch to load `calibration.json` next to the v3 bundle and pipe forecasts through `Calibration.apply(...)`. That change is intentionally **not** included here.

In [ ]:
import json
from pathlib import Path

V3_ROOT = Path("app/models/forecast_v3")
saved_paths = []
skipped = []

for (sid, h), calib in calibrations.items():
    slot_dir = V3_ROOT / sid / f"h{h}"
    if not slot_dir.exists():
        skipped.append((sid, h, str(slot_dir)))
        continue
    out_path = slot_dir / "calibration.json"
    out_path.write_text(
        json.dumps(calib.to_json(), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    saved_paths.append(out_path)

print(f"Saved {len(saved_paths)} calibration.json files.")
for p in saved_paths:
    print("  +", p)
if skipped:
    print(f"\nSkipped {len(skipped)} pairs (no v3 model directory yet):")
    for sid, h, where in skipped:
        print(f"  - {sid} h={h}: {where}")

## 4 — Inline diagnostics / ผลตรวจในสมุดงาน

### 4a — Per-pair MAE / bias table

In [ ]:
with pd.option_context("display.float_format", "{:0.3f}".format):
    display(summary)

### 4b — Per-station bar chart: raw vs calibrated MAE / กราฟแท่งเทียบราย station

In [ ]:
import matplotlib.pyplot as plt

by_station = summary.groupby("station_id")[["raw_mae", "cal_mae"]].mean()
stations = list(by_station.index)
x = np.arange(len(stations))
width = 0.38

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(x - width/2, by_station["raw_mae"].values, width, label="raw MAE", color="#888888")
ax.bar(x + width/2, by_station["cal_mae"].values, width, label="calibrated MAE", color="#2a7fbf")
ax.set_xticks(x)
ax.set_xticklabels(stations)
ax.set_ylabel("Mean Absolute Error (°C)")
ax.set_title("Raw vs Calibrated MAE per Station (avg over horizons)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 4c — Per-hour bias for BKK_01 / กราฟ bias รายชั่วโมงสำหรับ BKK_01

**EN.** We pick BKK_01 as a representative station and plot raw bias vs L3-corrected bias by local hour-of-day. L3 is the 4-bucket hour calibration; we fit it directly here on the BKK_01 fit-slice for this diagnostic, regardless of which level `select_calibration` chose for that pair (so you can see what the hour-bucket correction would have looked like).

**TH.** เลือก BKK_01 เป็น station ตัวแทน แล้ววาด bias รายชั่วโมงทั้งแบบ raw และหลังแก้ด้วย L3. ฟิต L3 ตรง ๆ จาก fit-slice ของ BKK_01 เพื่อให้เห็นผลของ hour-bucket calibration เปรียบเทียบกับ raw.

In [ ]:
REP_STATION = "BKK_01"

rep_fit = fit_df[fit_df.station_id == REP_STATION]
rep_full = val_df[val_df.station_id == REP_STATION]

if len(rep_fit) >= 30 and len(rep_full):
    l3 = fit_calibration(
        level="L3",
        y_true=rep_fit.y_true.values,
        y_pred=rep_fit.y_pred.values,
        station_ids=rep_fit.station_id.values,
        horizons=rep_fit.horizon_h.values,
        hours=rep_fit.hour_local.values,
    )
    rep_pred_l3 = l3.apply(
        rep_full.y_pred.values,
        rep_full.station_id.values,
        rep_full.horizon_h.values,
        rep_full.hour_local.values,
    )
    by_hr = pd.DataFrame({
        "hour_local": rep_full.hour_local.values,
        "raw_resid": rep_full.y_pred.values - rep_full.y_true.values,
        "l3_resid": rep_pred_l3 - rep_full.y_true.values,
    }).groupby("hour_local").mean()

    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.plot(by_hr.index, by_hr.raw_resid, marker="o", label="raw bias", color="#cc4444")
    ax.plot(by_hr.index, by_hr.l3_resid, marker="s", label="L3-corrected bias", color="#2a7fbf")
    ax.set_xlabel("local hour of day")
    ax.set_ylabel("mean residual (pred − true) [°C]")
    ax.set_title(f"Per-hour bias — {REP_STATION} — raw vs L3")
    ax.set_xticks(range(0, 24, 2))
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"Not enough data for {REP_STATION}; per-hour bias chart skipped.")

### 4d — Residual histogram for one (station, horizon) / ฮิสโตแกรม residual

**EN.** Pick the first available (station, horizon) pair (default BKK_01 h=24 if present) and overlay raw vs calibrated residuals.

**TH.** เลือก (station, horizon) แรกที่มีอยู่ (เริ่มที่ BKK_01 h=24 ถ้ามี) แล้ววางทับฮิสโตแกรมของ residual แบบ raw และหลัง calibrated.

In [ ]:
preferred = ("BKK_01", 24)
if preferred in calibrations:
    sid_h, h_h = preferred
elif calibrations:
    sid_h, h_h = next(iter(calibrations.keys()))
else:
    sid_h, h_h = None, None

if sid_h is not None:
    pair = val_df[(val_df.station_id == sid_h) & (val_df.horizon_h == h_h)]
    raw_resid = pair.y_pred.values - pair.y_true.values
    cal_resid = pair.y_pred_cal.values - pair.y_true.values

    fig, ax = plt.subplots(figsize=(8, 4.2))
    bins = np.linspace(
        float(np.nanmin(np.concatenate([raw_resid, cal_resid]))),
        float(np.nanmax(np.concatenate([raw_resid, cal_resid]))),
        50,
    )
    ax.hist(raw_resid, bins=bins, alpha=0.55, label="raw residual", color="#cc4444")
    ax.hist(cal_resid, bins=bins, alpha=0.55, label="calibrated residual", color="#2a7fbf")
    ax.axvline(0.0, color="black", linewidth=0.8)
    ax.set_xlabel("residual (pred − true) [°C]")
    ax.set_ylabel("count")
    ax.set_title(f"Residual histogram — {sid_h} h={h_h}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No calibrations were fit; residual histogram skipped.")

## 5 — Risk-recall sanity check / ตรวจสอบ recall ของ high-watch

**EN.** Calibration must not silently drop high-watch coverage. We define a high-watch event as `heat_index >= 40 °C` (the same threshold used in `app/core/risk.py`'s extreme-caution band). For each (station, horizon) pair we compute recall (= proportion of true high-watch hours where the prediction also crosses 40 °C). Render side-by-side: raw vs calibrated.

**TH.** การ calibrate ต้องไม่ลด recall ของ high-watch อย่างมีนัย. เรานิยาม high-watch เมื่อ `heat_index >= 40 °C` (ตรงกับขีดของ `app/core/risk.py`). คำนวณ recall ต่อ (station, horizon) ทั้งแบบ raw และ calibrated แล้วแสดงเทียบกัน.

In [ ]:
HIGH_WATCH_THRESHOLD = 40.0

def _recall(y_true, y_pred, thr=HIGH_WATCH_THRESHOLD):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    pos = y_true >= thr
    if pos.sum() == 0:
        return float("nan")
    return float((y_pred[pos] >= thr).mean())

recall_rows = []
for (sid, h), group in val_df.groupby(["station_id", "horizon_h"], sort=False):
    if (sid, h) not in calibrations:
        continue
    n_pos = int((group.y_true >= HIGH_WATCH_THRESHOLD).sum())
    if n_pos == 0:
        recall_rows.append({
            "station_id": sid, "horizon_h": h,
            "n_high_watch": 0,
            "raw_recall": float("nan"),
            "cal_recall": float("nan"),
            "recall_delta": float("nan"),
        })
        continue
    raw_r = _recall(group.y_true, group.y_pred)
    cal_r = _recall(group.y_true, group.y_pred_cal)
    recall_rows.append({
        "station_id": sid, "horizon_h": h,
        "n_high_watch": n_pos,
        "raw_recall": raw_r,
        "cal_recall": cal_r,
        "recall_delta": cal_r - raw_r,
    })

recall_df = pd.DataFrame(recall_rows).sort_values(["station_id", "horizon_h"]).reset_index(drop=True)
with pd.option_context("display.float_format", "{:0.3f}".format):
    display(recall_df)

valid = recall_df.dropna(subset=["raw_recall", "cal_recall"])
if len(valid):
    macro_raw = valid.raw_recall.mean()
    macro_cal = valid.cal_recall.mean()
    print(f"\nMacro-avg recall — raw: {macro_raw:.3f}    calibrated: {macro_cal:.3f}    Δ: {macro_cal-macro_raw:+.3f}")
    if macro_cal + 1e-6 < macro_raw - 0.05:
        print("WARNING: calibrated macro recall dropped by more than 0.05 vs raw — inspect per-pair table above.")
else:
    print("No high-watch events present in val — recall sanity check is non-informative.")

## 6 — Mini-report / สรุปย่อ

**EN.** Summary of this notebook:

1. Loaded val-fold predictions written by notebook 03 at `notebooks/colab/_artifacts/val_predictions/{station}__h{H}.parquet`.
2. For each (station, horizon) pair, ran `select_calibration(...)` on a 70 / 30 time-based fit/select split inside the val window, picking the level (L1/L2/L3/L4) with the lowest MAE.
3. Saved the chosen `Calibration` as JSON next to each v3 model: `app/models/forecast_v3/{station}/h{H}/calibration.json` (without touching `bundle.json`).
4. Reported raw-vs-calibrated MAE and bias per pair, drew per-station bar chart, per-hour bias chart for BKK_01, residual histogram for one pair, and high-watch (HI ≥ 40 °C) recall sanity table.

**Follow-up (out of scope):** add a small block in `app/ml/forecast/predict.py` that, when `calibration.json` exists in the v3 slot, loads it via `Calibration.from_json(...)` and post-processes `_bundle.hi_mean[-1]` through `Calibration.apply(...)` before constructing `ForecastPoint`.

Next notebook: **`07_evaluate.ipynb`** — final test-set evaluation, sortie holdout metrics, and risk-band confusion matrix.

**TH.** สรุป notebook นี้:

1. โหลดผล val ที่ notebook 03 บันทึกไว้ที่ `notebooks/colab/_artifacts/val_predictions/{station}__h{H}.parquet`
2. แต่ละ (station, horizon) ใช้ `select_calibration(...)` แบ่ง val เป็น 70/30 ตามเวลา เพื่อ fit/select level ที่ MAE ต่ำสุด (L1/L2/L3/L4)
3. เซฟเป็น JSON ที่ `app/models/forecast_v3/{station}/h{H}/calibration.json` โดย**ไม่แตะ** `bundle.json`
4. รายงาน MAE/bias เทียบ raw vs calibrated ต่อคู่, กราฟแท่งระดับ station, กราฟ bias รายชั่วโมงของ BKK_01, ฮิสโตแกรม residual หนึ่งคู่ และ recall ของ high-watch (HI ≥ 40 °C)

**ที่ต้องทำต่อ (อยู่นอกขอบเขต notebook นี้):** แก้ `app/ml/forecast/predict.py` ให้โหลด `calibration.json` (ถ้ามี) และส่ง prediction ผ่าน `Calibration.apply(...)` ก่อนสร้าง `ForecastPoint`

ขั้นถัดไป: **`07_evaluate.ipynb`** — ประเมินบน test holdout และทำ confusion matrix ของ risk band.